# UNIVERSIDADE FEDERAL FLUMINENSE

**TCE11616 - Introdução ao Aprendizado Profundo**

**Prof.: Diogo Ferreira de Lima Silva**

---
# AULA 07 — EVITANDO O SOBREAJUSTE
---

Na aula passada vimos o **sobreajuste**: a rede vai tão bem no treino que começa a "decorar" os exemplos, e o desempenho em dados novos piora. Isso é especialmente comum quando temos **poucos dados**, situação frequente em problemas reais.

Neste notebook, vamos provocar o sobreajuste de propósito e testar algumas formas simples de combatê-lo.

## 0. Importando as bibliotecas

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)      # fixa a semente: os resultados ficam iguais a cada execução

# cores usadas nos gráficos
LARANJA, AZUL, VERDE, VERMELHO, CINZA = "#E08D2B", "#1F7A8C", "#2E7D4F", "#B3151A", "#5B6472"

---
## 1. Os dados: progressão do diabetes
---

O conjunto **Diabetes**, do scikit-learn, tem dados de 442 pacientes: idade, sexo, índice de massa corporal, pressão arterial e 6 exames de sangue. O alvo é uma medida da **progressão da doença um ano depois**. É um problema de **regressão** com **poucos dados**.

In [ ]:
from sklearn.datasets import load_diabetes

diabetes = load_diabetes(as_frame=True)
diabetes.frame.head()

Seguimos a receita da aula passada: treino, validação e teste (60/20/20) e padronização. Desta vez padronizamos também o **alvo** $y$. Isso tem uma vantagem prática: um modelo que sempre "chuta a média" terá erro (MSE) próximo de **1**, o que nos dá uma régua fácil para comparar.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = diabetes.data.values
y = diabetes.target.values.reshape(-1, 1)

X_tr, X_resto, y_tr, y_resto = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_te, y_val, y_te = train_test_split(X_resto, y_resto, test_size=0.5, random_state=42)

pad_X = StandardScaler().fit(X_tr)       # aprende média e desvio no treino
pad_y = StandardScaler().fit(y_tr)

def para_tensores(X, y):
    return (torch.tensor(pad_X.transform(X), dtype=torch.float32),
            torch.tensor(pad_y.transform(y), dtype=torch.float32))

X_treino, Y_treino = para_tensores(X_tr, y_tr)
X_valid,  Y_valid  = para_tensores(X_val, y_val)
X_teste,  Y_teste  = para_tensores(X_te, y_te)

print("treino:", X_treino.shape, "| validação:", X_valid.shape, "| teste:", X_teste.shape)

A função de treino é a mesma da aula passada, agora com `nn.MSELoss` (regressão). Uma novidade: ela recebe o **otimizador** como argumento, para podermos testar o weight decay mais adiante.

In [ ]:
def treinar(modelo, otimizador, epocas=1000):
    perda_fn = nn.MSELoss()
    perdas_treino, perdas_valid = [], []

    for epoca in range(epocas):
        modelo.train()
        perda = perda_fn(modelo(X_treino), Y_treino)
        otimizador.zero_grad()
        perda.backward()
        otimizador.step()

        modelo.eval()
        with torch.no_grad():
            perda_val = perda_fn(modelo(X_valid), Y_valid)

        perdas_treino.append(perda.item())
        perdas_valid.append(perda_val.item())

    return perdas_treino, perdas_valid


def erro(modelo, X, Y):
    modelo.eval()
    with torch.no_grad():
        return nn.MSELoss()(modelo(X), Y).item()


def desenha_curvas(perdas_treino, perdas_valid, titulo):
    plt.figure(figsize=(8, 4))
    plt.plot(perdas_treino, color=LARANJA, linewidth=2, label="treino")
    plt.plot(perdas_valid, color=AZUL, linewidth=2, label="validação")
    plt.axhline(1.0, color=CINZA, linestyle=":", label="chutar a média")
    plt.ylim(0, 1.5); plt.xlabel("época"); plt.ylabel("erro (MSE)")
    plt.title(titulo); plt.legend(); plt.grid(alpha=0.3)
    plt.show()

---
## 2. Provocando o sobreajuste
---

Uma rede **grande** (duas camadas de 256 neurônios, mais de 68 mil parâmetros) para apenas 265 pacientes de treino:

In [ ]:
torch.manual_seed(42)
rede_grande = nn.Sequential(
    nn.Linear(10, 256), nn.ReLU(),
    nn.Linear(256, 256), nn.ReLU(),
    nn.Linear(256, 1),
)
otimizador = torch.optim.Adam(rede_grande.parameters(), lr=0.001)
perdas_treino, perdas_valid = treinar(rede_grande, otimizador)

desenha_curvas(perdas_treino, perdas_valid, "Rede grande, sem nenhuma proteção")
print(f"erro final -> treino: {perdas_treino[-1]:.3f} | validação: {perdas_valid[-1]:.3f}")

O retrato clássico do sobreajuste:

- o erro de **treino** vai praticamente a **zero**: a rede decorou os 265 pacientes;
- o erro de **validação** cai no começo, mas logo volta a **subir** e termina quase tão ruim quanto simplesmente "chutar a média" (linha pontilhada).

A seguir, quatro remédios.

---
## 3. Remédio 1: uma rede menor
---

Se a rede tem capacidade demais, a solução mais direta é usar uma rede **menor**: com menos parâmetros, ela não consegue decorar tudo.

In [ ]:
torch.manual_seed(42)
rede_pequena = nn.Sequential(nn.Linear(10, 8), nn.ReLU(), nn.Linear(8, 1))
otimizador = torch.optim.Adam(rede_pequena.parameters(), lr=0.001)
pt, pv = treinar(rede_pequena, otimizador)

desenha_curvas(pt, pv, "Rede pequena (8 neurônios)")
print(f"erro final -> treino: {pt[-1]:.3f} | validação: {pv[-1]:.3f}")

Agora as duas curvas ficam próximas: a rede não consegue mais decorar o treino, e o erro de validação fica bem menor que o da rede grande.

---
## 4. Remédio 2: parar mais cedo
---

Nas curvas da rede grande, o erro de validação teve um **mínimo** nas primeiras épocas. E se pararmos o treino ali? Essa técnica se chama **parada antecipada** (*early stopping*).

In [ ]:
melhor_epoca = int(np.argmin(perdas_valid)) + 1     # +1 porque a contagem começa em 0
print("época com menor erro de validação:", melhor_epoca)

Como fixamos a semente, podemos simplesmente **treinar de novo** a mesma rede grande pelo número ideal de épocas:

In [ ]:
torch.manual_seed(42)
rede_parada = nn.Sequential(nn.Linear(10, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 1))
otimizador = torch.optim.Adam(rede_parada.parameters(), lr=0.001)
treinar(rede_parada, otimizador, epocas=melhor_epoca)

print(f"erro de validação com parada antecipada: {erro(rede_parada, X_valid, Y_valid):.3f}")

---
## 5. Remédio 3: weight decay
---

Redes que decoram os dados costumam ter **pesos muito grandes**. O **weight decay** (decaimento dos pesos) puxa todos os pesos um pouquinho em direção a zero a cada passo, impedindo que cresçam demais.

No PyTorch, basta trocar o otimizador `Adam` por `AdamW` e escolher o argumento `weight_decay`. Quanto maior o valor, mais forte o efeito. Vamos testar alguns valores e escolher pela validação:

In [ ]:
for wd in [0.1, 1, 10]:
    torch.manual_seed(42)
    rede = nn.Sequential(nn.Linear(10, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 1))
    otimizador = torch.optim.AdamW(rede.parameters(), lr=0.001, weight_decay=wd)
    pt, pv = treinar(rede, otimizador)
    print(f"weight_decay = {wd:<4} | erro treino: {pt[-1]:.3f} | erro validação: {pv[-1]:.3f}")

Com weight decay mais forte, o erro de treino deixa de ir a zero (a rede não consegue mais decorar) e o erro de validação diminui. Escolhemos o valor com menor erro de validação:

In [ ]:
torch.manual_seed(42)
rede_wd = nn.Sequential(nn.Linear(10, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 1))
otimizador = torch.optim.AdamW(rede_wd.parameters(), lr=0.001, weight_decay=10)
pt, pv = treinar(rede_wd, otimizador)

desenha_curvas(pt, pv, "Rede grande com weight decay = 10")

---
## 6. Remédio 4: dropout
---

O **dropout** desliga (zera) aleatoriamente uma parte dos neurônios a cada passo do treino. Assim, a rede não pode depender de nenhum neurônio específico e é forçada a aprender padrões mais robustos, em vez de decorar.

Veja o efeito de uma camada `nn.Dropout(0.5)` (desliga 50% dos valores) em um vetor de uns:

In [ ]:
dropout = nn.Dropout(0.5)
uns = torch.ones(10)

dropout.train()       # modo treino: desliga valores ao acaso
print("treino   :", dropout(uns))
print("treino   :", dropout(uns), " <- outro sorteio")

dropout.eval()        # modo avaliação: não desliga nada
print("avaliação:", dropout(uns))

Duas observações:

- os valores que sobram são **multiplicados por 2** (= 1 / 0,5), para que a soma média continue a mesma;
- na **avaliação** o dropout é desligado. É por isso que usamos `modelo.train()` e `modelo.eval()` na função de treino: aqui eles fazem diferença de verdade.

Para usar, basta colocar `nn.Dropout` depois de cada ativação:

In [ ]:
torch.manual_seed(42)
rede_dropout = nn.Sequential(
    nn.Linear(10, 256), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(256, 1),
)
otimizador = torch.optim.Adam(rede_dropout.parameters(), lr=0.001)
pt, pv = treinar(rede_dropout, otimizador)

desenha_curvas(pt, pv, "Rede grande com dropout de 50%")

---
## 7. Comparando tudo
---

Reunimos os modelos e, **só agora**, medimos o erro no conjunto de teste. Incluímos também a régua de "chutar a média".

In [ ]:
modelos = {
    "rede grande, sem proteção": rede_grande,
    "rede pequena":              rede_pequena,
    "parada antecipada":         rede_parada,
    "weight decay":              rede_wd,
    "dropout":                   rede_dropout,
}

erro_media = nn.MSELoss()(torch.zeros_like(Y_teste), Y_teste).item()    # prever sempre 0 (a média padronizada)

print(f"{'modelo':<28} {'treino':>7} {'validação':>10} {'teste':>7}")
print("-" * 56)
print(f"{'chutar a média':<28} {'':>7} {'':>10} {erro_media:>7.3f}")
for nome, modelo in modelos.items():
    print(f"{nome:<28} {erro(modelo, X_treino, Y_treino):>7.3f} "
          f"{erro(modelo, X_valid, Y_valid):>10.3f} {erro(modelo, X_teste, Y_teste):>7.3f}")

O que a tabela mostra:

- a rede grande sem proteção tem o **menor** erro de treino e um dos **maiores** erros de teste: decorou, mas não aprendeu;
- todas as técnicas reduzem o erro de teste, mesmo tendo erro de treino **maior**. Errar um pouco mais no treino é o preço de generalizar melhor;
- neste problema, com poucos dados, a solução mais simples (uma rede pequena) funciona tão bem quanto as outras. **Vale sempre testar o modelo simples primeiro.**

---
## Exercícios
---

**1.** Na Seção 3, teste redes pequenas com 2, 8 e 32 neurônios. Qual tem o menor erro de validação?

**2.** Na Seção 6, troque o dropout de 0.5 para 0.2. O efeito fica mais fraco ou mais forte?

**3.** Combine duas técnicas: rede grande com dropout de 0.5 **e** `AdamW` com `weight_decay=1`. O erro de validação melhora?

**4.** Remova a linha `modelo.eval()` da função `treinar` e treine de novo a rede com dropout. O que acontece com a curva de validação? Por quê?